In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run util path

In [0]:
dbutils.widgets.text("catalog", "ecommerce", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path=f'dataset path/{data_source}'
print(base_path)

In [0]:
df_bronze= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.display(10)

In [0]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col_name in date_columns:
    df_bronze = df_bronze.withColumn(
        col_name,
        F.to_date(F.to_timestamp(col_name, "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"))
    )
df_bronze.display()


In [0]:
display(
    df_bronze.filter(
    
        (F.col("order_status") == "delivered") &
        (F.col("order_delivered_customer_date").isNull())
    

)
)

In [0]:
 df_clean = df_bronze.filter(
    ~(
        (F.col("order_status") == "delivered") &
        (F.col("order_delivered_customer_date").isNull())
    )
)

In [0]:
df_clean.count()

In [0]:
df_duplicates= df_clean.groupBy('order_id').count().filter('count > 1')
df_duplicates.display()

In [0]:
df_duplicates= df_clean.groupBy('customer_id').count().filter('count > 1')
df_duplicates.display()

In [0]:
df_silver=df_clean

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver=spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")

df_gold= df_silver.select('order_id','customer_id', 'order_status','order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date')

In [0]:
df_gold=df_gold.withColumnRenamed('order_delivered_customer_date','delivered_date')
df_gold=df_gold.withColumnRenamed('order_estimated_delivery_date','estimated_delivery_date')
df_gold=df_gold.withColumnRenamed('order_purchase_timestamp','purchase_date')
df_gold.show(2)

                                  

In [0]:
df = df_gold.withColumn(
    "delivery_days",
    F.datediff(
        F.col("delivered_date"),
        F.col("purchase_date")
    )
)
df.show(2)

In [0]:
df = df.withColumn(
    "is_late_delivery",
    (F.col("delivered_date") >
     F.col("estimated_delivery_date")).cast("int")
)

In [0]:
df.display()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")